# Notebook — Label-Efficiency SỬA — PHẦN B (10%, 25%, 50%, 100%)**Đây là 1 trong 2 notebook chạy song song.** Phần B xử lý fraction **10/25/50/100%** (1 seed=42 mỗi mức, như thiết kế gốc). Chạy phần này trên một tài khoản Kaggle; phần A (1%, 5%) chạy trên tài khoản kia.Ước tính thời gian phần B trên T4 đơn: **~6,5–7 giờ** (an toàn dưới 12h).**Sửa lỗi cốt lõi:** I-JEPA nạp **encoder SSL sạch** (NB03, chưa thấy nhãn RSNA) + head ngẫu nhiên, KHÔNG dùng checkpoint đã fine-tune. Mọi pipeline khác giữ y hệt notebook gốc.**Input cần attach:** (1) ảnh RSNA .dcm; (2) metadata rsna_train/val/test.csv (đúng bản cũ); (3) output NB03 chứa `ijepa_vit_small_nih_50k_best_encoder.pth`; (4) bật Internet.**Output phần này:** `label_eff_B_high_results.csv`, `label_eff_B_high_preds.npz`, `label_eff_B_high_summary.csv`.

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os, gc, time, random, copy, json, math
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm
try: import pydicom
except ImportError: os.system('pip install -q pydicom'); import pydicom
from PIL import Image
try: import timm
except ImportError: os.system('pip install -q timm'); import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_AMP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| timm', timm.__version__)
IMG_SIZE=224; IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]
NOTEBOOK_START=time.time()

Device: cuda | timm 1.0.26


In [2]:
# ============================================================
# CELL 2: TÌM INPUT — và GUARD chặn checkpoint fine-tune
# ============================================================
WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT   = Path('/kaggle/input')

def find_file(filename):
    for root in [WORKING_ROOT, INPUT_ROOT]:
        hits=list(root.rglob(filename))
        if hits: return hits[0]
    return None

RSNA_TRAIN_CSV=find_file('rsna_train.csv')
RSNA_VAL_CSV  =find_file('rsna_val.csv')
RSNA_TEST_CSV =find_file('rsna_test.csv')

# --- Encoder SSL sạch (ưu tiên 50k best_encoder, như Notebook 04 đã làm) ---
def find_ssl_encoder():
    pats=['ijepa_vit_small_nih_50k_best_encoder.pth',
          'ijepa_vit_small_nih_30k_best_encoder.pth',
          'ijepa_vit_small_nih_*_best_encoder.pth',
          'ijepa_vit_small_nih_*_encoder_epoch_*.pth']
    for pat in pats:
        for root in [INPUT_ROOT, WORKING_ROOT]:
            hits=sorted(root.rglob(pat))
            if hits: return hits[-1]
    return None

SSL_ENCODER_CKPT=find_ssl_encoder()
print('RSNA Train :', RSNA_TRAIN_CSV)
print('RSNA Val   :', RSNA_VAL_CSV)
print('RSNA Test  :', RSNA_TEST_CSV)
print('SSL encoder:', SSL_ENCODER_CKPT)

assert RSNA_TRAIN_CSV and RSNA_VAL_CSV and RSNA_TEST_CSV, 'Thiếu CSV metadata'
assert SSL_ENCODER_CKPT is not None, 'Không tìm thấy encoder SSL Notebook 03 (ijepa_*_best_encoder.pth)'
# GUARD: tuyệt đối không dùng checkpoint đã fine-tune RSNA
assert 'finetune' not in SSL_ENCODER_CKPT.name.lower(), \
    'LỖI: đây là checkpoint đã fine-tune RSNA — sẽ rò rỉ nhãn. Hãy dùng *_best_encoder.pth của NB03.'
print('\nGUARD OK — đang dùng encoder SSL sạch (chưa thấy nhãn RSNA).')

# Lập chỉ mục ảnh để fix path
_all=[p for root in [INPUT_ROOT] for p in list(root.rglob('*.dcm'))+list(root.rglob('*.png'))]
FNAME2PATH={p.name:str(p) for p in _all}
print('Ảnh lập chỉ mục:', len(FNAME2PATH))

RSNA Train : /kaggle/input/notebooks/nguyentongphuc/01-eda-metadata-split-nih-rsna/metadata/rsna_train.csv
RSNA Val   : /kaggle/input/notebooks/nguyentongphuc/01-eda-metadata-split-nih-rsna/metadata/rsna_val.csv
RSNA Test  : /kaggle/input/notebooks/nguyentongphuc/01-eda-metadata-split-nih-rsna/metadata/rsna_test.csv
SSL encoder: /kaggle/input/notebooks/nguyentongphuc/03-i-jepa-pretraining-2/notebook03_ijepa_pretrain/checkpoints/ijepa_vit_small_nih_50k_best_encoder.pth

GUARD OK — đang dùng encoder SSL sạch (chưa thấy nhãn RSNA).
Ảnh lập chỉ mục: 29692


In [3]:
# ============================================================
# CELL 3: DATASET + TRANSFORMS (khớp notebook cũ)
# ============================================================
def fix_df_paths(df):
    df=df.copy()
    def _fix(p):
        p=str(p)
        if Path(p).exists(): return p
        return FNAME2PATH.get(Path(p).name, None)
    df['image_path']=df['image_path'].apply(_fix)
    return df.dropna(subset=['image_path']).reset_index(drop=True)

def read_image(path):
    p=str(path)
    if p.endswith('.dcm') or p.endswith('.dicom'):
        ds=pydicom.dcmread(p); arr=ds.pixel_array.astype(np.float32)
        if str(getattr(ds,'PhotometricInterpretation','MONOCHROME2')).upper()=='MONOCHROME1':
            arr=arr.max()-arr
        mn,mx=arr.min(),arr.max(); arr=((arr-mn)/(mx-mn+1e-8)*255).astype(np.uint8)
        return Image.fromarray(arr).convert('RGB')
    return Image.open(p).convert('RGB')

train_transform=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.RandomHorizontalFlip(0.5),
    T.RandomRotation(7),T.ColorJitter(brightness=0.1,contrast=0.1),T.ToTensor(),
    T.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
eval_transform=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.ToTensor(),
    T.Normalize(IMAGENET_MEAN,IMAGENET_STD)])

class RSNADataset(Dataset):
    def __init__(self,df,transform=None): self.df=df.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; img=read_image(row['image_path'])
        if self.transform: img=self.transform(img)
        return img, torch.tensor(float(row['label']),dtype=torch.float32)
print('Dataset OK')

Dataset OK


In [4]:
# ============================================================
# CELL 4: ĐỌC METADATA + POS_WEIGHT + VAL/TEST LOADER
# ============================================================
train_df=fix_df_paths(pd.read_csv(RSNA_TRAIN_CSV))
val_df  =fix_df_paths(pd.read_csv(RSNA_VAL_CSV))
test_df =fix_df_paths(pd.read_csv(RSNA_TEST_CSV))
n_neg=(train_df['label']==0).sum(); n_pos=(train_df['label']==1).sum()
POS_WEIGHT=torch.tensor([n_neg/n_pos],dtype=torch.float32)
print(f'Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}')
print(f'POS_WEIGHT = {POS_WEIGHT.item():.4f}')   # kỳ vọng 3.4387
val_loader =DataLoader(RSNADataset(val_df, eval_transform),batch_size=32,shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(RSNADataset(test_df,eval_transform),batch_size=32,shuffle=False,num_workers=2,pin_memory=True)

Train 18,678 | Val 4,003 | Test 4,003
POS_WEIGHT = 3.4387


In [5]:
# ============================================================
# CELL 5: I-JEPA CLASSIFIER + LOAD ENCODER SSL SẠCH (mắt xích đã sửa)
# ============================================================
class IJEPAClassifier(nn.Module):
    def __init__(self, encoder, embed_dim=384, dropout=0.1):
        super().__init__()
        self.encoder=encoder
        self.classifier=nn.Sequential(nn.LayerNorm(embed_dim),nn.Dropout(dropout),nn.Linear(embed_dim,1))
    def forward(self,x): return self.classifier(self.encoder(x)).squeeze(1)

def load_ijepa_ssl(dropout=0.1):
    '''
    SỬA: chỉ nạp ENCODER SSL (encoder_state_dict / student_encoder_state_dict)
    từ checkpoint pretrain NB03. Head LUÔN khởi tạo ngẫu nhiên — KHÔNG nạp head
    đã train. Đây là điểm khác biệt với notebook cũ (vốn nạp cả model đã fine-tune).
    '''
    enc=timm.create_model('vit_small_patch16_224',pretrained=False,num_classes=0)
    ck=torch.load(SSL_ENCODER_CKPT,map_location='cpu',weights_only=False)
    sd=(ck.get('encoder_state_dict') or ck.get('student_encoder_state_dict') or ck)
    missing,unexpected=enc.load_state_dict(sd,strict=False)
    model=IJEPAClassifier(enc,embed_dim=enc.num_features,dropout=dropout)
    for p in model.parameters(): p.requires_grad=True
    return model.to(DEVICE)

# Sanity: head phải là ngẫu nhiên (khác nhau giữa 2 lần gọi nếu seed khác)
_m=load_ijepa_ssl(); _w=_m.classifier[-1].weight.detach().abs().mean().item()
print(f'I-JEPA SSL nạp OK | encoder từ {SSL_ENCODER_CKPT.name} | head random (|w|~{_w:.4f})')
del _m; gc.collect()

@torch.no_grad()
def predict_probs(model,loader):
    model.eval(); probs,lbls=[],[]
    for imgs,labels in loader:
        imgs=imgs.to(DEVICE)
        with torch.amp.autocast(_AMP_DEVICE,enabled=torch.cuda.is_available()):
            logits=model(imgs)
        logits=logits.squeeze(-1) if logits.dim()>1 else logits
        probs.extend(torch.sigmoid(logits).float().cpu().numpy()); lbls.extend(labels.numpy())
    return np.array(lbls).astype(int), np.array(probs)
def quick_auc(model,loader):
    y,p=predict_probs(model,loader); return roc_auc_score(y,p)

# Templates baseline (download 1 lần)
print('Downloading ImageNet templates...')
_vit_template=timm.create_model('vit_small_patch16_224',pretrained=True,num_classes=1).cpu()
_r50_template=tvm.resnet50(weights='IMAGENET1K_V2'); _r50_template.fc=nn.Linear(_r50_template.fc.in_features,1); _r50_template=_r50_template.cpu()
print('Templates OK')

I-JEPA SSL nạp OK | encoder từ ijepa_vit_small_nih_50k_best_encoder.pth | head random (|w|~0.0257)


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


Templates OK


In [6]:
# ============================================================
# CELL 6: TRAIN_QUICK (KHỚP notebook cũ — không đổi)
# ============================================================
IJEPA_HEAD_LR=3e-5; IJEPA_ENC_LR_LOW=None; IJEPA_ENC_LR=3e-6
R50_LR=1e-4; WD=0.05; ACCUM=2
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_quick(model,sub_df,num_epochs,head_lr,enc_lr=None,wd=0.05,accum=2):
    criterion=nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
    if enc_lr is not None and hasattr(model,'encoder') and hasattr(model,'classifier'):
        param_groups=[{'params':[p for p in model.encoder.parameters() if p.requires_grad],'lr':enc_lr},
                      {'params':[p for p in model.classifier.parameters() if p.requires_grad],'lr':head_lr}]
    elif enc_lr is None and hasattr(model,'encoder') and hasattr(model,'classifier'):
        for p in model.encoder.parameters(): p.requires_grad=False
        param_groups=[{'params':[p for p in model.classifier.parameters() if p.requires_grad],'lr':head_lr}]
    else:
        param_groups=[{'params':[p for p in model.parameters() if p.requires_grad],'lr':head_lr}]
    optimizer=torch.optim.AdamW(param_groups,weight_decay=wd)
    scaler=torch.amp.GradScaler(_AMP_DEVICE,enabled=torch.cuda.is_available())
    loader=DataLoader(RSNADataset(sub_df,train_transform),batch_size=16,shuffle=True,num_workers=2,pin_memory=True)
    best_val=-1
    for epoch in range(num_epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step,(imgs,labels) in enumerate(loader):
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            with torch.amp.autocast(_AMP_DEVICE,enabled=torch.cuda.is_available()):
                logits=model(imgs); logits=logits.squeeze(-1) if logits.dim()>1 else logits
                loss=criterion(logits,labels)/accum
            scaler.scale(loss).backward()
            if (step+1)%accum==0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],0.5)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        if (step+1)%accum!=0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],0.5)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        v=quick_auc(model,val_loader)
        if v>best_val: best_val=v
    return best_val
print('train_quick OK')

train_quick OK


In [7]:
# ============================================================
# CELL 7: CHẠY LABEL-EFFICIENCY HỢP LỆ — I-JEPA từ encoder SSL sạch
# ============================================================
OUTPUT_DIR=Path('/kaggle/working/label_eff_B_high'); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
SEED_CONFIG={0.01:{'seeds':[42,123,456,789,1234],'epochs':20},
             0.05:{'seeds':[42,123,456],'epochs':15},
             0.10:{'seeds':[42],'epochs':15}, 0.25:{'seeds':[42],'epochs':15},
             0.50:{'seeds':[42],'epochs':15}, 1.00:{'seeds':[42],'epochs':15}}
ALL_FRACTIONS=[0.10,0.25,0.50,1.00]   # PHẦN B

def sample_sub_df(frac,seed):
    set_seed(seed)
    pos=train_df[train_df['label']==1].sample(frac=frac,random_state=seed)
    neg=train_df[train_df['label']==0].sample(frac=frac,random_state=seed)
    return pd.concat([pos,neg]).sample(frac=1,random_state=seed).reset_index(drop=True)

results=[]; pred_store={}
for frac in ALL_FRACTIONS:
    cfg=SEED_CONFIG[frac]; seeds,epochs=cfg['seeds'],cfg['epochs']
    ijepa_enc_lr = IJEPA_ENC_LR_LOW if frac<=0.05 else IJEPA_ENC_LR
    if (time.time()-NOTEBOOK_START)/3600>11.0: print('Time guard'); break
    print('\n'+'='*60); print(f'{int(frac*100)}% labels | {len(seeds)} seeds x {epochs}ep | enc_lr={ijepa_enc_lr}'); print('='*60)
    for seed in seeds:
        sub_df=sample_sub_df(frac,seed)
        print(f'  seed={seed} | n={len(sub_df)} (pos={(sub_df.label==1).sum()},neg={(sub_df.label==0).sum()})')

        # --- I-JEPA SSL sạch (head random) ---
        set_seed(seed)
        m=load_ijepa_ssl(dropout=0.1)
        vb=train_quick(m,sub_df,epochs,IJEPA_HEAD_LR,ijepa_enc_lr)
        yt,yp=predict_probs(m,test_loader); ta=roc_auc_score(yt,yp)
        results.append({'model':'I-JEPA SSL (clean)','fraction':frac,'seed':seed,'n_train':len(sub_df),'val_auc':round(vb,6),'test_auc':round(ta,6)})
        pred_store[f'IJEPA-SSL|{frac}|{seed}']=(yt,yp)
        print(f'    I-JEPA SSL   val={vb:.4f} test={ta:.4f}')
        del m; gc.collect(); torch.cuda.empty_cache()

        # --- ResNet50 fresh ---
        set_seed(seed); r=copy.deepcopy(_r50_template).to(DEVICE)
        vb=train_quick(r,sub_df,epochs,R50_LR,enc_lr=None)
        yt,yp=predict_probs(r,test_loader); ta=roc_auc_score(yt,yp)
        results.append({'model':'ResNet50 ImageNet','fraction':frac,'seed':seed,'n_train':len(sub_df),'val_auc':round(vb,6),'test_auc':round(ta,6)})
        print(f'    ResNet50     val={vb:.4f} test={ta:.4f}')
        del r; gc.collect(); torch.cuda.empty_cache()

        # --- ViT-Small fresh ---
        set_seed(seed); v=copy.deepcopy(_vit_template).to(DEVICE)
        vb=train_quick(v,sub_df,epochs,R50_LR,enc_lr=None)
        yt,yp=predict_probs(v,test_loader); ta=roc_auc_score(yt,yp)
        results.append({'model':'ViT-Small ImageNet','fraction':frac,'seed':seed,'n_train':len(sub_df),'val_auc':round(vb,6),'test_auc':round(ta,6)})
        print(f'    ViT-Small    val={vb:.4f} test={ta:.4f}')
        del v; gc.collect(); torch.cuda.empty_cache()

        pd.DataFrame(results).to_csv(OUTPUT_DIR/'label_eff_B_high_results.csv',index=False)

results_df=pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR/'label_eff_B_high_results.csv',index=False)
np.savez_compressed(OUTPUT_DIR/'label_eff_B_high_preds.npz',
    **{k.replace('|','__'):np.stack(v) for k,v in pred_store.items()})
print('\nDONE'); display(results_df)


10% labels | 1 seeds x 15ep | enc_lr=3e-06
  seed=42 | n=1868 (pos=421,neg=1447)
    I-JEPA SSL   val=0.7899 test=0.7809
    ResNet50     val=0.8485 test=0.8164
    ViT-Small    val=0.8607 test=0.8342

25% labels | 1 seeds x 15ep | enc_lr=3e-06
  seed=42 | n=4670 (pos=1052,neg=3618)
    I-JEPA SSL   val=0.8026 test=0.7948
    ResNet50     val=0.8693 test=0.8337
    ViT-Small    val=0.8673 test=0.8497

50% labels | 1 seeds x 15ep | enc_lr=3e-06
  seed=42 | n=9339 (pos=2104,neg=7235)
    I-JEPA SSL   val=0.8139 test=0.7964
    ResNet50     val=0.8780 test=0.8376
    ViT-Small    val=0.8734 test=0.8614

100% labels | 1 seeds x 15ep | enc_lr=3e-06
  seed=42 | n=18678 (pos=4208,neg=14470)
    I-JEPA SSL   val=0.8164 test=0.8076
    ResNet50     val=0.8877 test=0.8557
    ViT-Small    val=0.8811 test=0.8675

DONE


,model,fraction,seed,n_train,val_auc,test_auc
0,I-JEPA SSL (clean),0.10,42,1868,0.789924,0.780869
1,ResNet50 ImageNet,0.10,42,1868,0.848502,0.816419
2,ViT-Small ImageNet,0.10,42,1868,0.860663,0.834185
3,I-JEPA SSL (clean),0.25,42,4670,0.802572,0.794768
4,ResNet50 ImageNet,0.25,42,4670,0.869347,0.833692
5,ViT-Small ImageNet,0.25,42,4670,0.867284,0.849674
6,I-JEPA SSL (clean),0.50,42,9339,0.813916,0.796367
7,ResNet50 ImageNet,0.50,42,9339,0.878042,0.837633
8,ViT-Small ImageNet,0.50,42,9339,0.873400,0.861394
9,I-JEPA SSL (clean),1.00,42,18678,0.816356,0.807604


In [8]:
# ============================================================
# CELL 8: TỔNG HỢP + SO VỚI SỐ CŨ (bị rò rỉ) — PHẦN B (10-100%)
# ============================================================
agg=results_df.groupby(['model','fraction'])['test_auc'].agg(['mean','std','count']).reset_index()
agg['std']=agg['std'].fillna(0.0)
print('=== KẾT QUẢ HỢP LỆ (encoder SSL sạch) — Test AUC ===')
for _,r in agg.iterrows():
    print(f"  {r['model']:<20} {int(r['fraction']*100):>3}%  AUC={r['mean']:.4f} (n={int(r['count'])})")

# I-JEPA Full v2 cũ (bị rò rỉ) tại 10-100% — từ Bảng 4.11
OLD_LEAKED={0.10:0.7830, 0.25:0.7955, 0.50:0.7997, 1.00:0.8137}
print('\n=== I-JEPA: SỐ MỚI (sạch) vs SỐ CŨ (rò rỉ nhãn) — 10..100% ===')
for frac in [0.10,0.25,0.50,1.00]:
    row=agg[(agg.model=='I-JEPA SSL (clean)')&(agg.fraction==frac)]
    if len(row):
        mu=float(row['mean'].iloc[0])
        print(f"  {int(frac*100):>3}%: sạch={mu:.4f}  |  cũ(rò rỉ)={OLD_LEAKED[frac]:.4f}  |  Δ={mu-OLD_LEAKED[frac]:+.4f}")
print('\nGHI CHÚ: ở vùng nhãn đủ (10-100%), encoder đã được fine-tune nhiều nên')
print('chênh lệch sạch-vs-cũ thường NHỎ hơn so với vùng 1-5%. Phần rò rỉ gây méo')
print('mạnh nhất ở vùng cực ít nhãn (xem Phần A).')
agg.to_csv(OUTPUT_DIR/'label_eff_B_high_summary.csv',index=False)
print('Đã lưu summary.')

=== KẾT QUẢ HỢP LỆ (encoder SSL sạch) — Test AUC ===
  I-JEPA SSL (clean)    10%  AUC=0.7809 (n=1)
  I-JEPA SSL (clean)    25%  AUC=0.7948 (n=1)
  I-JEPA SSL (clean)    50%  AUC=0.7964 (n=1)
  I-JEPA SSL (clean)   100%  AUC=0.8076 (n=1)
  ResNet50 ImageNet     10%  AUC=0.8164 (n=1)
  ResNet50 ImageNet     25%  AUC=0.8337 (n=1)
  ResNet50 ImageNet     50%  AUC=0.8376 (n=1)
  ResNet50 ImageNet    100%  AUC=0.8557 (n=1)
  ViT-Small ImageNet    10%  AUC=0.8342 (n=1)
  ViT-Small ImageNet    25%  AUC=0.8497 (n=1)
  ViT-Small ImageNet    50%  AUC=0.8614 (n=1)
  ViT-Small ImageNet   100%  AUC=0.8675 (n=1)

=== I-JEPA: SỐ MỚI (sạch) vs SỐ CŨ (rò rỉ nhãn) — 10..100% ===
   10%: sạch=0.7809  |  cũ(rò rỉ)=0.7830  |  Δ=-0.0021
   25%: sạch=0.7948  |  cũ(rò rỉ)=0.7955  |  Δ=-0.0007
   50%: sạch=0.7964  |  cũ(rò rỉ)=0.7997  |  Δ=-0.0033
  100%: sạch=0.8076  |  cũ(rò rỉ)=0.8137  |  Δ=-0.0061

GHI CHÚ: ở vùng nhãn đủ (10-100%), encoder đã được fine-tune nhiều nên
chênh lệch sạch-vs-cũ thường NHỎ hơn so

## Gộp với Phần ASau khi cả hai notebook chạy xong, tải về `label_eff_A_low_results.csv` (phần A) và `label_eff_B_high_results.csv` (phần B). Gộp:```pythonimport pandas as pda=pd.read_csv('label_eff_A_low_results.csv')b=pd.read_csv('label_eff_B_high_results.csv')full=pd.concat([a,b]).sort_values(['fraction','model','seed'])full.to_csv('label_eff_FULL_results.csv',index=False)summary=full.groupby(['model','fraction'])['test_auc'].agg(['mean','std','count'])print(summary)```Bảng `summary` chính là Bảng 4.10 + 4.11 phiên bản **hợp lệ, không rò rỉ** để thay vào báo cáo. Gửi tôi file `label_eff_FULL_results.csv` để tôi giúp viết lại RQ2 + tính DeLong nếu cần.